# 最高到達accuracyを取得する

In [1]:
from datetime import date

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import json
import re
import sys
from collections import defaultdict

project_root = os.path.join(os.getcwd(), "..")
result_dir = os.path.join(project_root, "results")


def load_file_and_extract_max_accs(date, lr, target_concept_config_name, model_size, result_dir):
    """各initVecTypeとseedに対して、回したepochs中の最高到達accuracyを抽出する関数"""
    # if target_concept_config_type != '':
    #     target_concept_config_type = '-' + target_concept_config_type
    # dirname_pattern1 = f"gemma-3-{model_size}B-lr0.01{target_concept_config_type}-{date}-hidden_layer(-?\\d+)-seed(\\d+)_target_concepts_initvecwith(.*)"
    # dirname_pattern2 = f"gemma-3-{model_size}B-lr0.01{target_concept_config_type}-{date}-seed(\\d+)_target_concepts_initvecwith(.*)"

    target_concept_config_name = target_concept_config_name.replace('.json', '')
    dirname_pattern1 = f"gemma-3-{model_size}B-lr{lr}-{date}-hidden_layer(-?\\d+)-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"
    dirname_pattern2 = f"gemma-3-{model_size}B-lr{lr}-{date}-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"
    initVecType_to_seed_to_maxacc = defaultdict(dict)
    initVecType_to_seed_to_epoch_at_maxacc = defaultdict(dict)
    for dirname in os.listdir(result_dir):

        if re.match(dirname_pattern1, dirname):
            match_groups = re.search(dirname_pattern1, dirname)
            layer_idx = int(match_groups.group(1))
            seed = int(match_groups.group(2))
            initVecType = f"{match_groups.group(3)}_layer{layer_idx}"
        elif re.match(dirname_pattern2, dirname):
            match_groups = re.search(dirname_pattern2, dirname)
            print(match_groups)
            seed = int(match_groups.group(1))
            initVecType = match_groups.group(2)
        else:
            if date in dirname:
                # print(dirname)
                pass
            continue  # ディレクトリ名がどちらのパターンにもマッチしない場合はスキップ

        # print(f"Processing directory: {dirname}, initVecType: {initVecType}, seed: {seed}")
        
        if not os.path.exists(os.path.join(result_dir, dirname, 'logit_score_summary.json')):
            # print(f"Warning: logit_score_summary.json not found in {dirname}. Skipping.")
            continue
        with open(os.path.join(result_dir, dirname, 'logit_score_summary.json'), "r") as f:
            concept_to_epoch_to_scores = json.load(f)

        # 各conceptについて、回したepochs中の最高到達accuracyとそのepochを抽出
        max_accs = []
        epochs_at_maxacc = []
        for epoch_to_scores in concept_to_epoch_to_scores.values():
            max_accuracy = 0
            max_epoch_at_maxacc = 0
            for epoch, scores in epoch_to_scores.items():
                # print(f"Epoch: {epoch}, Accuracy: {scores['accuracy']}")
                if scores['accuracy'] > max_accuracy:
                    max_accuracy = scores['accuracy']
                    max_epoch_at_maxacc = epoch
            max_accs.append(max_accuracy)
            epochs_at_maxacc.append(max_epoch_at_maxacc)
        # print(f"Max Accuracies: {max_accs}")
        # print(f"Epochs at Max Accuracies: {epochs_at_maxacc}")
        initVecType_to_seed_to_maxacc[initVecType][seed] = max_accs
        initVecType_to_seed_to_epoch_at_maxacc[initVecType][seed] = epochs_at_maxacc
    return initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc


## 12B

In [2]:

date = '20260323'
model_size = '12'

target_concept_config_name = 'target_concepts_mini.json'


In [3]:
lr = 0.003
initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)

for initVecType, seed_to_maxacc in initVecType_to_seed_to_maxacc.items():
    for seed, max_accs in seed_to_maxacc.items():
        mean_max_acc = np.mean(max_accs)
        std_max_acc = np.std(max_accs)
        print(f"InitVecType: {initVecType:15}, Seed: {seed}, Mean Max Accuracy: {mean_max_acc:.4f}, Std: {std_max_acc:.4f}")
print()
for initVecType, seed_to_epoch_at_maxacc in initVecType_to_seed_to_epoch_at_maxacc.items():
    for seed, epochs_at_maxacc in seed_to_epoch_at_maxacc.items():
        mean_epoch_at_maxacc = np.mean([int(e) for e in epochs_at_maxacc])
        std_epoch_at_maxacc = np.std([int(e) for e in epochs_at_maxacc])
        print(f"InitVecType: {initVecType:15}, Seed: {seed}, Mean Epoch number at Max Accuracy: {mean_epoch_at_maxacc:.2f}, Std: {std_epoch_at_maxacc:.2f}")


InitVecType: otherCatCent_by_GlbPrimDebiasedHS_lam0005_layer10, Seed: 0, Mean Max Accuracy: 0.6252, Std: 0.1594
InitVecType: CatCent_by_GlbPrimDebiasedHSMixed_layer10, Seed: 0, Mean Max Accuracy: 0.6264, Std: 0.1603
InitVecType: otherCatCent_by_GlbPrimDebiasedHSMixed_layer11, Seed: 0, Mean Max Accuracy: 0.6329, Std: 0.1385
InitVecType: otherCatCent_by_GlbPrimDebiasedHS_lam0005_layer11, Seed: 0, Mean Max Accuracy: 0.6941, Std: 0.1322
InitVecType: CatCent_by_GlbPrimDebiasedHSMixed_layer11, Seed: 0, Mean Max Accuracy: 0.6453, Std: 0.1264
InitVecType: otherCatCent_by_GlbPrimDebiasedHSMixed_layer10, Seed: 0, Mean Max Accuracy: 0.6190, Std: 0.1601
InitVecType: CatCent_by_GlbPrimDebiasedHS_lam0005_layer10, Seed: 0, Mean Max Accuracy: 0.6156, Std: 0.1486
InitVecType: CatCent_by_GlbPrimDebiasedHS_lam0005_layer11, Seed: 0, Mean Max Accuracy: 0.6215, Std: 0.1498

InitVecType: otherCatCent_by_GlbPrimDebiasedHS_lam0005_layer10, Seed: 0, Mean Epoch number at Max Accuracy: 1.70, Std: 0.78
InitVecType

In [4]:
not_show_initVecTypes = [
    'zero',
    'category_COG'
]

initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(date, model_size, result_dir)

for initVecType, seed_to_maxacc in initVecType_to_seed_to_maxacc.items():
    if initVecType in not_show_initVecTypes:
        continue
    for seed, max_accs in seed_to_maxacc.items():
        mean_max_acc = np.mean(max_accs)
        std_max_acc = np.std(max_accs)
        print(f"InitVecType: {initVecType:15}, Seed: {seed}, Mean Max Accuracy: {mean_max_acc:.4f}, Std: {std_max_acc:.4f}")
print()
for initVecType, seed_to_epoch_at_maxacc in initVecType_to_seed_to_epoch_at_maxacc.items():
    if initVecType in not_show_initVecTypes:
        continue
    for seed, epochs_at_maxacc in seed_to_epoch_at_maxacc.items():
        mean_epoch_at_maxacc = np.mean([int(e) for e in epochs_at_maxacc])
        std_epoch_at_maxacc = np.std([int(e) for e in epochs_at_maxacc])
        print(f"InitVecType: {initVecType:15}, Seed: {seed}, Mean Epoch number at Max Accuracy: {mean_epoch_at_maxacc:.2f}, Std: {std_epoch_at_maxacc:.2f}")


TypeError: load_file_and_extract_max_accs() missing 2 required positional arguments: 'model_size' and 'result_dir'

In [23]:
for initVecType, seed_to_maxacc in initVecType_to_seed_to_maxacc.items():
    for seed, max_accs in seed_to_maxacc.items():
        print(f"InitVecType: {initVecType:15}, Seed: {seed}, Max Accuracies: {[round(acc, 2) for acc in max_accs]}")

InitVecType: category_COG   , Seed: 0, Max Accuracies: [0.35, 0.23, 0.58, 0.43, 0.58, 0.22, 0.24, 0.3, 0.42, 0.52, 0.23, 0.36, 0.5, 0.34, 0.35, 0.54, 0.37, 0.51, 0.54, 0.52, 0.51, 0.5, 0.54, 0.34, 0.36, 0.56, 0.31, 0.33, 0.44, 0.39, 0.68, 0.52, 0.43, 0.74, 0.32, 0.59, 0.48, 0.48, 0.78, 0.4, 0.39, 0.77, 0.3, 0.5, 0.34, 0.56, 0.44, 0.79, 0.31, 0.54, 0.33, 0.53, 0.68, 0.44, 0.45, 0.46, 0.51, 0.41, 0.37, 0.24, 0.37, 0.46, 0.71, 0.47, 0.55, 0.62, 0.47, 0.38, 0.66, 0.27, 0.36, 0.4, 0.55, 0.31, 0.45, 0.39, 0.44, 0.43, 0.36, 0.29, 0.4, 0.33, 0.75, 0.35, 0.49, 0.22, 0.56, 0.54, 0.4, 0.4, 0.53, 0.45, 0.52, 0.48, 0.36, 0.49, 0.25, 0.53, 0.33, 0.48]
InitVecType: other_category_COG, Seed: 0, Max Accuracies: [0.37, 0.26, 0.51, 0.4, 0.56, 0.41, 0.21, 0.27, 0.58, 0.6, 0.21, 0.33, 0.52, 0.34, 0.53, 0.41, 0.42, 0.41, 0.72, 0.67, 0.44, 0.43, 0.42, 0.34, 0.4, 0.59, 0.31, 0.28, 0.66, 0.3, 0.45, 0.48, 0.43, 0.67, 0.33, 0.56, 0.4, 0.43, 0.89, 0.51, 0.36, 0.57, 0.39, 0.47, 0.34, 0.33, 0.53, 0.61, 0.31, 0.62, 

### 固定epoch時点でのacc比較